# CSCI 410/1411 Homework 8: Tabular Reinforcement Learning

## Assignment Overview

Welcome to Homework 8!

This week, you will be writing code to play Blackjack, a two-player card game with inherent stochasticity, since you never know what card will be dealt next. If you've been to a casino, you know that Blackjack is played between a gambler, called the player, and a dealer, or the "house". The player's strategy is unrestricted; but the dealer is confined to play according a known strategy. As a result, the dealer is not really an adversarial agent, so the player should not optimize by searching for a minimax strategy (or, more precisely, an expected minimax strategy, since the environment is stochastic). Instead, the dealer's (fixed) behavior can be incorporated into the environment. From this point of view, a gambler should not view Blackjack as a game, but rather as a sequential decision process: i.e., a stochastic process that transitions randomly from state to state, for which he seeks an optimal action at each state. Indeed, in this assignment, you will model Blackjack as a *Markov* decision process.

## The Blackjack Environment

We will be using the [gymnasium](https://gymnasium.farama.org/) library (abbreviated Gym) in this assignment.

Gym was built to facilitate research on reinforcement learning (RL) algorithms. It simulates myriad environments—modeled as Markov decision processes—and thereby enables AI researchers to experiment with the latest and the greatest RL algorithms on established benchmarks.

You will not be interacting with Gym directly. Instead, you will be interacting with a `Blackjack` class that we wrote for you, which extends an `MDP`, which inherits from Gym. The `Blackjack` and `MDP` classes are defined in `mdp.py`.

Here are some of the more relevant functions to this assignment:

`gym.make`: Initializes our Gym environment. This object is passed into the `Blackjack` constructor.

`mdp.reset`: Resets the environment to a new start state. The Blackjack start state is random, as it depends on the cards dealt.

`mdp.step`: Takes a "step" in the environment. This means starting off at a state $s$, taking an action $a$, and returning the next state $s'$. In Blackjack, the player can choose to hit or stand. After making his choice, his hand or the dealer's hand changes.

`mdp.get_transition_prob`: Given a state and an action taken in that state, gets the next possible states in our environment and their corresponding probabilities. For example, given the dealer's card, the player's hand, and the usable ace bit, if the player chooses to hit, this function will return all the possible next states that can arise when the player gets another card. It also computes the probability of entering each of these states.

`mdp.get_reward`: Returns the reward associated with a state. For example, if the player busts by going over 21, he loses the game, so his reward is $-1$. In our environment, a player either receives $+1$ for winning, $-1$ for losing, or 0 if the game is a draw.

Run the code cell below to setup your notebook.

In [ ]:
from IPython.display import display
import PIL.Image
import matplotlib.pyplot as plt

import numpy as np
import gymnasium as gym
from typing import List, Dict, Tuple, Optional
import itertools
import math
import time

from mdp import Blackjack, BlackjackState, BlackjackAction, BlackjackPolicy
from utils import render_gym_blackjack_env, Q_table_to_policy, plot_value_function, plot_policy, plot_observation_count, plot_policy_with_observation

#### Initial States

Run the code below to see an example of an initial state. Run it several times to see several examples!

As outlined above, the state comprises the player's total value, a bit indicating whether the player has a usable ace or not, and the dealer's visible card. If you do not see "usable ace" depicted as part of the state, the player does not have a usable ace.

**Hint:** You may have to run the code multiple times before you see a useable ace.

In [ ]:
env = gym.make('Blackjack-v1', render_mode='rgb_array') # initializes the Blackjack environment in Gymnasium
state = env.reset() # resets the environment to the starting state (dealing hands to the player and dealer)
rgb_array = env.render() # load in the environment as an rgb array
image = PIL.Image.fromarray(rgb_array)
image = image.resize((image.width // 2, image.height // 2))
image.save('blackjack.png')
display(image) # display the image for our notebook from our modified rgb array
env.close()

#### Gameplay

Run the code below to watch a simulation of a game of Blackjack.

**N.B.** You can alter the parameter to `time.sleep()` line to speed up / slow down the game.

In [ ]:
env = gym.make('Blackjack-v1', render_mode='rgb_array')

def play_blackjack() -> None:
    """
    Simulate a round of Blackjack with a simple naive policy.
    
    This demonstration function initializes the environment and follows a basic strategy:
    - Hit if player total < 20
    - Stand if player total >= 20
    
    The function renders each step and displays the final outcome.
    """
    # Setting up our environment and starting state
    state = env.reset()[0]
    print("Initial State:")
    render_gym_blackjack_env(env, state)
    time.sleep(1)

    # running the game until the player or dealer busts
    done = False
    while not done:
        # Naive policy: stand if 20 or higher, otherwise hit
        action = 0 if state[0] >= 20 else 1
        state, reward, done, info, _ = env.step(action)
        action_str = 'Stand' if action == 0 else 'Hit'
        print(f"Action taken: {action_str}")
        render_gym_blackjack_env(env, state)
        
        time.sleep(5) # Feel free to modify this value to slow down / speed up the game
        
    # displaying the final state and rewards
    print("Final State:")
    render_gym_blackjack_env(env, state)

    if reward > 0:
        print("Player wins!")
        print(f"Reward: {reward}")
    elif reward < 0:
        print("Player loses.")
        print(f"Reward: {reward}")
    else:
        print("It's a draw.")
        print(f"Reward: {reward}")

play_blackjack()

### Data Structures

The code below is excerpted from `mdp.py`. It describes how we represent the key components of the Blackjack MDP and its solution: e.g., the states, the actions, the policies, etc. Be sure you understand these data structures before proceeding, since you will not be able to correctly compute with the data they store otherwise.

In [ ]:
BlackjackState = Tuple[int, int, int] # player hand total, dealer's card value, either 0 or 1 depending on the presence of a usable ace
BlackjackAction = int # either 0 to stand or 1 to hit
BlackjackPolicy = Dict[BlackjackState, BlackjackAction] # mapping from each state to an action

ValueFunction = Dict[BlackjackState, float] # for Part 3, a table that maps each state to a value

QTable = Dict[Tuple[BlackjackState, BlackjackAction], float] # for Part 4, a table that maps each state-action pair to a Q-value

## Part 1: Conceptual Questions

#### Win Probabilities

Please include the correct answer for each question in your README.

**1.** What are the probabilities that the player will win/draw/lose if the player stands on a 19 and the dealer's cards total **16**, and the dealer does not have an ace? For simplicity, you may assume cards are drawn with replacement.

**Recall** The dealer draws until her total is at least 17, so with this hand, she must draw another card from the deck.

A. Win: 8/13, Draw: 1/13, Lose: 4/13  
B. Win: 10/13, Draw: 1/13, Lose: 2/13  
C. Win: 9/13, Draw: 2/13, Lose: 2/13  
D. Win: 11/13, Draw: 1/13, Lose: 1/13  

Use your answer to problem 1 and Bellman's equation to solve the next two problems:

**2.** What are the probabilities that the player will win/draw/lose if the player stands on a 19 and the dealer's cards total **15**, and the dealer does not have an ace?

A. Win: 127/169, Draw: 14/169, Lose: 28/169  
B. Win: 117/169, Draw: 14/169, Lose: 38/169  
C. Win: 10/13, Draw: 1/13, Lose: 2/13  
D. Win: 9/13, Draw: 1/13, Lose: 3/13

**3.** What are the probabilities that the player will win/draw/lose if the player stands on a 19 and the dealer's cards total **14**, and the dealer does not have an ace?

A. Win: 127/169, Draw: 14/169, Lose: 28/169  
B. Win: 8/13, Draw: 1/13, Lose: 4/13  
C. Win: 1609/2197, Draw: 196/2197, Lose: 392/2197  
D. Win: 10/13, Draw: 1/13, Lose: 2/13

Recall the definition of the **value function** at a state $s$, given a policy $\pi$: it is the expected value of the long-term rewards of following policy $\pi$ starting at $s$. The **optimal value function** is the expected value of the long-term rewards of following an optimal policy.

**4.** Assume a win yields reward $+1$, a loss reward $-1$, and a draw reward $0$, which of the following is a valid interpretation of the optimal value function for Blackjack in terms of win/draw/lose probabilities?

A. The probability of winning from a given state.  
B. The probability of winning, accounting for ties.  
C. The probability of winning minus the probability of losing.  
D. The expected number of hands the player can expect to play before going broke.  

It is possible to compute the win/draw/lose probabilities for all states, not just when the player has been dealt a 19. But computing these probabilities is tedious! Lukcily, there is a simple and intuitive alternative to direct computation, namely to *simulate* sample episodes from each start state of interest.

## Part 2: Monte Carlo Simulation

The [Monte Carlo Method](https://en.wikipedia.org/wiki/Monte_Carlo_method) can be used to simulate randomness. As such, it is applicable to simulating casino games, such as the Gambler's Ruin and Blackjack.

Recall the Gambler's Ruin example of a Markov chain from lecture. It is possible to use Monte Carlo simulation to estimate the state values in the Gambler's Ruin Markov chain, by simulating many many episodes from an initial state $s$—most of which terminate with value 0 and a few with value 1—and then declaring the average of the rewards earned across all episodes as the estimated value of state $s$. We can estimate the values of the states of our Blackjack MDP, given a fixed policy, in the exact same way.

Implement the `monte_carlo_simulation` function to evaluate the `always_stand_policy`, whereby a player always stands and never hits. This policy is useful as a baseline.

**Hint:** Use the `reset` method to start an episode and the `step` method to step through it until it ends. These methods can be found in the `Blackjack` class in `mdp.py`. 

In [ ]:
def monte_carlo_simulation(mdp: Blackjack, policy: BlackjackPolicy, num_episodes: int) -> Tuple[float, float]:
    """
    Run a Monte Carlo simulation on a Blackjack MDP using a specified policy.
    
    This function evaluates a policy by running many episodes and computing
    statistics on the outcomes. Each episode follows the given policy until
    the game terminates, then records the final reward.

    Args:
        mdp: An initialized Blackjack MDP with methods like reset and step for gameplay
        policy: A policy mapping states to actions in the Blackjack environment
        num_episodes: Number of episodes to simulate for statistical reliability

    Returns:
        Tuple containing:
        - win_rate: The proportion of episodes won (wins / num_episodes)
        - average_reward: The average reward across all episodes
    """
    win_rate = 0
    average_reward = 0
    
    num_wins = 0
    num_losses = 0
    num_draws = 0
    
    # TODO: Implement Monte Carlo Simulations here

    print(f"Played {num_episodes} episodes.")
    print(f"Win Rate: {win_rate:.2%}")
    print(f"Average Reward: {average_reward:.2f}")
    print(f"Wins: {num_wins}, Losses: {num_losses}, Draws: {num_draws}")

    return win_rate, average_reward

In [ ]:
def create_always_stand_policy(mdp: Blackjack) -> BlackjackPolicy:
    """
    Create a policy where the player always stands (never hits).
    
    This conservative policy serves as a baseline for comparison.
    The player takes no additional cards regardless of the current state.
    
    Args:
        mdp: Blackjack MDP instance to get all possible states
        
    Returns:
        Policy dictionary mapping all states to action 0 (stand)
    """
    policy = {}
    states = mdp.get_all_states()
    action = 0  # 0 is stand, 1 is hit
    for state in states:
        policy[state] = action
    return policy

Always standing is *not* a very effective policy. You can expect to achieve a win rate of about 38% with this policy.


In [ ]:
mdp = Blackjack(gym.make('Blackjack-v1', render_mode='rgb_array'))
always_stand_policy = create_always_stand_policy(mdp)

_, _ = monte_carlo_simulation(mdp, always_stand_policy, 100000)

## Part 3: Value Iteration

Your next task is to implement value iteration, and then to use it to solve for an optimal policy in our Blackjack MDP.

Implement the `value_iteration` function, and compare the performance of the policy it learns to the `always_stand_policy`.

**Value Iteration**

1. **Initialize**: Initialize the value function $V(s)$, for all states $s$. E.g., initialize $V(s)$ to zero, for all states $s$.

2. **Repeat until convergence:**

   1. For all states $s$ and actions $a$, $Q(s,a) \gets r(s,a) + \gamma \sum_{s' \in S} P[s' | s, a] V(s')$
   1. For all states $s$, $V(s) \gets \max_{a \in A} Q(s,a)$

<!--
2. **Iterative Update**: For each state $s$, update the value function using the Bellman equation:
   $$
   V(s) = \max_{a \in \mathcal{A}(s)} \sum_{s'} P(s'|s,a) \left[ R(s,a,s') + \gamma V(s') \right]
   $$
   where $\mathcal{A}(s)$ is the set of possible actions in state $s$, $P(s'|s,a)$ is the transition probability, $R(s,a,s')$ is the reward, and $\gamma$ is the discount factor. -->

3. **Policy Extraction**: Extract a policy $\pi(s)$ from the value function $V(s)$ by choosing a maximizing action:
   $$
   \pi^*(s) \in \arg\max_{a \in \mathcal{A}} Q(s,a)
   $$

In [ ]:
def compare_value_functions(V1: ValueFunction, V2: ValueFunction) -> float:
    """
    Compute the L2 norm between two value functions by state.

    This function is typically used to compare value functions after an iteration
    of Value Iteration to check for convergence.

    Args:
        V1: First value function
        V2: Second value function

    Returns:
        L2 norm of the difference between the value functions
    """
    diff = 0
    for state in V1:
        diff += ((V1[state] - V2[state]) ** 2)
    return math.sqrt(diff)

def value_iteration(mdp: Blackjack, threshold: float = 1e-5, max_iter: int = 100) -> Tuple[ValueFunction, BlackjackPolicy]:
    """
    Perform value iteration to learn an optimal value function for Blackjack.
    
    Value iteration repeatedly applies the Bellman optimality equation:
    V(s) = max_a Σ P(s'|s,a) [R(s,a,s') + γV(s')]
    
    Since Blackjack is episodic with γ=1, the equation simplifies to:
    V(s) = max_a Σ P(s'|s,a) [R(s') + V(s')] for non-terminal states
    V(s) = R(s) for terminal states

    Args:
        mdp: A Blackjack MDP that has been initialized with transition and reward functions
        threshold: Convergence threshold for L2 norm between successive value functions
        max_iter: Maximum number of iterations to prevent infinite loops

    Returns:
        Tuple containing:
        - V: Optimal value function mapping each state to its expected value
        - pi: Optimal policy derived from the value function
    """
    actions = (0, 1)
    states = mdp.get_all_states()
    V: ValueFunction = {}
    for state in states:
        V[state] = 0.0
    pi: BlackjackPolicy = {}


    # TODO: Implement Value Iteration Here

    return V, pi

In [ ]:
mdp = Blackjack(gym.make('Blackjack-v1', render_mode='rgb_array'))
V, pi = value_iteration(mdp = mdp)

You should expect to achieve a win rate of about 43% with the policy learned by value iteration.

In [ ]:
_, _ = monte_carlo_simulation(mdp, pi, 100000)

Next, we provide you with code that enables you to visualizing the value function and the corresponding policy!

In [ ]:
# PLOT VI VALUE FUNCTION, WITH & WITHOUT AN ACE
plot_value_function(V)

In [ ]:
# PLOT VI POLICY, WITH & WITHOUT AN ACE
plot_policy(pi)

## Part 4: Q-Learning

Value iteration is a dynamic programming algorithm that requires a complete traversal of an MDP's state space. As such, it is not scalable to very large state spaces. An alternative is reinforcement learning (RL), which learns by simulating episodes in the environment.

$Q$-learning is a simple RL algorithm. For as many simulated episodes as desired, an agent takes an action $a$ at a state $s$, receives a reward $r$ and transitions to a new state $s'$, and then updates its $Q$ value at the state-action pair $(s, a)$ based on the reward and the transition. Specifically, assuming learning rate $\alpha$,
$$
Q(s, a) \gets (1 - \alpha) Q(s, a) + \alpha [r + \max_{a' \in A} Q(s', a')]
$$

$Q$-learning is an off-policy learning algorithm, meaning it is not concerned with the rewards it accrues while learning. As a result, it can take arbitrary actions while learning, rather than focus on high-valued actions.

Implement the `q_learning` function and compare the performance of this third policy with the other two you've already implemented, the `always_stand_policy` and an optimal policy computed via value iteraton..

In [ ]:
def q_learning(mdp: Blackjack, num_episodes: int, alpha: float) -> Tuple[QTable, Dict[Tuple[BlackjackState, BlackjackAction], int]]:
    """
    Perform the Q-learning algorithm to learn a value function for Blackjack.
    
    Q-learning is a model-free reinforcement learning algorithm that learns
    the optimal action-value function Q*(s,a) through experience.
    
    The update rule is:
    Q(s,a) ← (1-α)Q(s,a) + α[r + max_a' Q(s',a')]
    
    This implementation uses ε-greedy exploration (purely random for simplicity).

    Args:
        mdp: A Blackjack MDP for environment interaction
        num_episodes: Number of episodes (games) to run for learning
        alpha: Learning rate controlling how quickly Q-values are updated

    Returns:
        Tuple containing:
        - Q: Q-table mapping (state, action) pairs to Q-values
        - observation_counts: Dictionary tracking visits to each (state, action) pair
    """
    Q: QTable = {}
    observation_counts = {} 
    actions = (0, 1)
    states = mdp.get_all_states()

    # Initialize Q-table and observation counts
    for state_action_pair in list(itertools.product(states, actions)):
        Q[state_action_pair] = 0.0
        observation_counts[state_action_pair] = 0

    # TODO: Implement Q-Learning

    
    return Q, observation_counts

In [ ]:
mdp = Blackjack(gym.make('Blackjack-v1', render_mode='rgb_array'))
Q, observation_counts = q_learning(mdp = mdp, num_episodes=100000, alpha=0.01)
pi = Q_table_to_policy(Q)

You should expect to achieve a win rate of about 43% with the policy learned from Q-learning.


In [ ]:
_, _ = monte_carlo_simulation(mdp, pi, 100000)

Again, we provide you with code that enables you to visualizing the value function and the corresponding policy!

In [ ]:
plot_policy(pi)

### Visualizing how often we visit each state in Q-Learning!

In [ ]:
plot_observation_count(observation_counts)

In [ ]:
plot_policy_with_observation(observation_counts, pi)

## Part 5: Conceptual Questions

Please include the correct answer for each question in your README.

#### Blackjack in Casinos

The game of Blackjack you just solved is a relatively simple version of the game. It is *not* the version played in casinos. The game played in casinos has the following additional components:

1. Before any cards are dealt, the player decides how much money to bet.
2. A normal win pays out $2 for every $1 the player bets. Being dealt Blackjack (the player's total is 21 in exactly two cards) pays $3 for every $1 the player bets.
3. Players have two additional actions they may take:
   - **Double:** Double the bet and draw only one additional card.
   - **Split:** If dealt two cards of the same rank, double the bet and split the cards into two separate hands.
4. Blackjack hands can be played sequentially without replacing previously played cards. After a certain number of hands, the decks are refreshed and shuffled.

**1.** Which of the following best describes how the Blackjack MDP should be modified to incorporate these casino rules?

A. Keep the same states and actions, but increase all rewards to account for betting.  
B. Expand the states to track bets, hand information, and split status; add double and split actions; and update rewards and transitions accordingly.  
C. Replace the states with the player's total winnings and keep the original actions.  
D. Keep the same MDP but increase the discount factor to account for betting.

**2.** How would these modifications affect value iteration and Q-learning?

A. Value iteration would no longer require transition probabilities because the state space is larger.  
B. Both algorithms would run much faster because there are more possible actions, making it easier to find an optimal policy.  
C. Both algorithms would become more computationally demanding, and value iteration would require a larger explicit transition model.  
D. Neither algorithm would be affected because the optimal policy is independent of the state space size.

#### Card Counting

An alternative to tracking every card that has been played is to add a [count](http://en.wikipedia.org/wiki/Card_counting) value to the state. The count helps keep track of how many high-valued cards remain in the deck compared to low-valued cards. When a high card (10, J, Q, K, Ace) is played, the count decreases by 1. When a low card (2, 3, 4, 5, 6) is played, the count increases by 1. The remaining cards (7, 8, 9) are neutral, and do not affect the count.

When the count is high, so that there are more high-valued cards in the deck, the player's chances of winning improves. Why? For two reasons: 1. with many face cards in the deck, the player is more likely to be dealt a Blackjack; and 2. the dealer, whose policy is fixed, is more likely to bust.

**3.** How should the Blackjack MDP be modified to incorporate card counting?

A. Track only the player's current total and ignore cards played in previous hands.  
B. Assume that every card is replaced after it is drawn, so the deck composition remains constant.  
C. Add a running card count to the state, update it as cards are revealed, and account for cards being drawn without replacement.  
D. Reset the card count to zero after every time the dealer deals a new round of cards.  

**4.** If the MDP tracks the player's total, dealer information, and a running card count, does this state representation necessarily satisfy the Markov property?

A. Yes, because the running count uniquely determines which cards remain in the deck.  
B. No, because different remaining deck compositions can have the same count but different probabilities for the next card.  
C. No, because any process involving randomness violates the Markov property.  
D. Yes, because including a running count makes all transitions deterministic.  

**5.** Given that exact transition probabilities are difficult to calculate when counting cards, which of value iteration and Q-learning would be easier to implement with card counting, and why?

A. Value iteration, because it does not need transition probabilities.  
B. Value iteration, because it learns directly from gameplay without a model.  
C. Both algorithms require exact transition probabilities, so neither would be easier.  
D. Q-learning, because it learns from sampled transitions without requiring explicit transition probabilities.  

**6.** Suppose we tried to solve for an optimal policy using the full Blackjack state representation, including the exact remaining deck composition, and we tried again using only a running card count. How would you expect their performance to compare?

A. The full-state policy would perform better because it has access to more information.  
B. The card-counting policy would perform better because it has fewer states, so it could compute/learn something closer to optimal for its representation.  
C. Both policies would perform equally well because they use the same actions.  
D. The full-state policy would perform worse because its larger state space makes its optimal decisions less effective.